# AIRPATH-AI Milestone 3D — end-to-end forecast + spatial validation

This notebook reproduces the development-only comparison of:

1. forecast-only station PM2.5;
2. oracle IDW p=1 from observed peer stations;
3. forecast+IDW p=1 from forecasted peer stations.

It uses persisted V1 validation predictions generated from training-period fits. It does not retrain models, use the exposed forecasting test period for selection, calculate exposure, or optimize routes. Route-segment references are hourly oracle-IDW pseudo-references—not road measurements.

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.end_to_end_validation import generate_validation_outputs

In [ ]:
output_directory = PROJECT_ROOT / "data/processed/end_to_end_validation"
outputs = generate_validation_outputs(
    prediction_csv=PROJECT_ROOT / "data/processed/xgboost_forecasting_predictions.csv",
    network_path=PROJECT_ROOT / "data/processed/road_network/healthyair_pilot_osm.json.gz",
    output_directory=output_directory,
    report_path=PROJECT_ROOT / "reports/end_to_end_validation.md",
)
outputs["metrics"].loc[outputs["metrics"]["aggregation"].eq("pooled")]

In [ ]:
display(outputs["metrics"].loc[outputs["metrics"]["aggregation"].eq("per_horizon")])
display(outputs["decomposition"])

In [ ]:
display(outputs["mapping_metrics"])
display(
    outputs["route_sample"][[
        "mode",
        "segment_index",
        "eta",
        "mapped_target_time",
        "oracle_spatial_pm25",
        "forecast_spatial_pm25",
        "absolute_error",
        "reliability_status",
    ]]
)

In [ ]:
display(outputs["correlations"])
display(outputs["status_metrics"])
outputs["decision"]

In [ ]:
for filename in (
    "heldout_pipeline_mae.png",
    "mapping_sensitivity.png",
    "reliability_error_proxy.png",
):
    display(Image(filename=output_directory / filename))

## Interpretation boundary

Current HealthyAir validation is hourly. Ceiling, floor, and nearest mappings select an observed/predicted hourly target without interpolation. None validates PM2.5 at an exact minute-level arrival time.

The readiness decision permits only a restricted offline exposure-aggregation experiment. It does not authorize route recommendation or optimization, and station 5's negative combined R² remains a material limitation.